# Yeoubi: train a ~100M model on a Colab T4

Trains on the parts written by `colab_prepare_data.ipynb`. One run spans many sessions:
every session does Runtime > Run all, and the notebook resumes from the newest
checkpoint in `CKPT_REPO` on Hugging Face and picks the matching data parts.

- Runtime: **T4 GPU** (Runtime > Change runtime type).
- Needs a Colab secret named `HF_TOKEN` (key icon in the left sidebar) holding a
  Hugging Face token with write access, with notebook access enabled.
- Keep the settings cell identical between sessions of the same run.

In [ ]:
# SPDX-License-Identifier: BigCode-OpenRAIL-M-1.0
REPO = "https://github.com/Fury7425/Yeoubi.git"
DATA = "hf"   # where the parts live: "hf" (uses no Drive space) or "drive"
HF_REPO = ""   # dataset repo holding the parts, such as "your-name/yeoubi-data"
DRIVE_DATA = "/content/drive/MyDrive/yeoubi_data"   # only when DATA == "drive"
CKPT_REPO = ""   # private model repo for checkpoints, such as "your-name/yeoubi-ckpt"
EPOCHS = 1    # passes over the data. Sets max_steps, which the LR and A8->A4 schedules
              # depend on, so never change it in the middle of a run.
WINDOW = 24   # parts fetched and decompressed per session, about 26 MB to download and
              # 66 MB on disk each. One part is about 1557 steps, roughly 40 minutes on
              # a T4, so 24 covers a 12 h session with room to spare.
KEEP = 2      # resumable checkpoints kept in CKPT_REPO, roughly 1 GB each

In [ ]:
import glob
import gzip
import json
import os
import random
import re
import shutil
import threading
import time

assert CKPT_REPO, "set CKPT_REPO"
assert DATA == "drive" or HF_REPO, "set HF_REPO when DATA is 'hf'"

if not os.path.isdir("/content/Yeoubi"):
    !git clone --depth 1 {REPO} /content/Yeoubi
%cd /content/Yeoubi
!pip -q install -r requirements.txt
!nvidia-smi -L

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    print("no HF_TOKEN secret: the checkpoint repo calls below will fail")

from huggingface_hub import HfApi, hf_hub_download

api = HfApi()
api.create_repo(CKPT_REPO, private=True, exist_ok=True)

if DATA == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
LOCAL_CKPT = "/content/ckpt"
os.makedirs(LOCAL_CKPT, exist_ok=True)

In [ ]:
def step_of(path):
    match = re.search(r"step_(\d+)", os.path.basename(path))
    return int(match.group(1)) if match else -1


def is_resumable(name):
    # The pre-A4 snapshot and final.pt are kept for reference, never resumed from or pruned.
    return name.startswith("step_") and name.endswith(".pt") and "pre_a4" not in name


def fetch(name):
    """Local path of one file from the prepared data set."""
    if DATA == "drive":
        return f"{DRIVE_DATA}/{name}"
    return hf_hub_download(HF_REPO, name, repo_type="dataset")


def gunzip(name, target, seed=None):
    # train.py's loader dispatches on .jsonl / .pt and does not read .gz itself.
    with gzip.open(fetch(name), "rb") as source:
        lines = source.readlines()   # one part fits in RAM by design
    if seed is not None:
        # A part holds documents in the order the builder received them, and its tail
        # comes from whichever source was still being read. Seeded, so a resumed session
        # reads the part in the same order.
        random.Random(seed).shuffle(lines)
    with open(target, "wb") as output:
        output.writelines(lines)


newest = max(filter(is_resumable, api.list_repo_files(CKPT_REPO)), key=step_of, default="")
resume = hf_hub_download(CKPT_REPO, newest) if newest else ""
step = step_of(newest) if newest else 0

with open(fetch("meta.json"), encoding="utf-8") as handle:
    meta = json.load(handle)
parts, steps_per_part = meta["parts"], meta["steps_per_part"]
# scripts/pack_for_colab.py writes no total_steps.
max_steps = meta.get("total_steps", parts * steps_per_part) * EPOCHS

# The loader position is not checkpointed, so the checkpoint step picks the first part.
# Workers interleave neighbouring files, so a resume replays or skips about one part.
first = (step // steps_per_part) % parts
window_parts = [(first + index) % parts for index in range(min(WINDOW, parts))]
if DATA == "hf":
    # Fetched one at a time, a part took about 10 s (2.5 MB/s measured), which is minutes
    # of GPU session spent waiting. This fills the local cache 8 files at a time; the
    # fetch() calls below then find every file already there.
    from huggingface_hub import snapshot_download
    snapshot_download(
        HF_REPO,
        repo_type="dataset",
        max_workers=8,
        allow_patterns=[f"parts/part_{part:05d}.jsonl.gz" for part in window_parts] + ["val.jsonl.gz"],
    )
shutil.rmtree("/content/train", ignore_errors=True)
os.makedirs("/content/train")
os.makedirs("/content/val", exist_ok=True)
for index, part in enumerate(window_parts):
    # Named by position, not part number: the loader sorts paths, and a window that
    # wraps past the last part must still be read in window order.
    gunzip(f"parts/part_{part:05d}.jsonl.gz", f"/content/train/{index:03d}.jsonl", seed=part)
gunzip("val.jsonl.gz", "/content/val/val.jsonl")
print(f"resume={newest or 'none'} step={step}/{max_steps} parts {window_parts[0]}..{window_parts[-1]}")

In [ ]:
_sizes = {}
_sync_lock = threading.Lock()   # the background loop and the final sync must not overlap


def sync_checkpoints(stable_only=True):
    """Upload finished local checkpoints to CKPT_REPO and drop old ones there."""
    with _sync_lock:
        uploaded = False
        for path in sorted(glob.glob(f"{LOCAL_CKPT}/*.pt"), key=step_of):
            size = os.path.getsize(path)
            # torch.save writes in place, so a file is only known complete once its
            # size has stopped changing between two polls.
            if stable_only and _sizes.get(path) != size:
                _sizes[path] = size
                continue
            api.upload_file(
                path_or_fileobj=path, path_in_repo=os.path.basename(path), repo_id=CKPT_REPO
            )
            os.remove(path)   # only after the upload committed
            uploaded = True
        if not uploaded:
            return
        resumable = sorted(filter(is_resumable, api.list_repo_files(CKPT_REPO)), key=step_of)
        for old in resumable[:-KEEP]:
            api.delete_file(old, CKPT_REPO)
        if len(resumable) > KEEP:
            # A deleted file stays in the repo history and keeps counting toward the
            # account's storage; squashing the history is what releases it.
            api.super_squash_history(CKPT_REPO)


def _sync_loop():
    while True:
        time.sleep(60)
        try:
            sync_checkpoints()
        except Exception as error:   # a network hiccup must not end the syncing
            print("checkpoint sync failed, will retry:", error)


if "_sync_thread" not in globals():
    _sync_thread = threading.Thread(target=_sync_loop, daemon=True)
    _sync_thread.start()

In [ ]:
# --max_seq_len 4096 and batch 4 x accum 4 = 16 chunks per step are what meta.json's
# step counts assume (MAX_SEQ_LEN / CHUNKS_PER_STEP in colab_prepare_data.ipynb).
# hidden 768 / ffn 2048 with the default RDT layer counts is 109M parameters.
# --bf16 false: a T4 has no native bf16, fp16 is the fast path. If the loss turns NaN,
# switch to --bf16 true (slower) rather than lowering the learning rate first.
# --compile_model false: on the first T4 session torch.compile spent over 8 minutes
# recompiling for each new batch length, hit its recompile limit in the byte-patch
# encoder and fell back to eager anyway. Every resumed session would pay that again.
# --gradient_checkpointing false: with it on, the first T4 session ran 4.1 s per step
# (929,577 steps = about 44 days). Off skips recomputing activations in the backward
# pass, expected 20-30% faster but unmeasured. If this runs out of GPU memory, set it
# back to true; that changes speed only, not the schedule, so a resume is safe.
# --save_every 500: the default 2500 (about 2.5 h on a T4) lost a whole session when
# Colab disconnected at step 2450. A dead runtime gives no warning, so the interval is
# the most work a disconnect can cost: 500 is about 30 minutes. Each save is a ~1 GB
# upload in the background thread; KEEP and the history squash cap the repo size.
# If the window runs out before the session ends, train.py loops over the same parts
# again; raise WINDOW on a faster GPU.
!python -u train.py \
    --train_data "/content/train/*.jsonl" --val_data "/content/val/*.jsonl" \
    --output_dir {LOCAL_CKPT} --resume_from "{resume}" --max_steps {max_steps} \
    --max_seq_len 4096 --batch_size 4 --grad_accum_steps 4 \
    --hidden_dim 768 --n_heads 12 --ffn_dim 2048 \
    --bf16 false --compile_model false --gradient_checkpointing false \
    --save_every 500 --num_workers 2 --wandb_project ""

sync_checkpoints(stable_only=False)   # train.py has exited, so every file is complete